# 🔄 Versioning & Lifecycle Management

This lab demonstrates S3 object versioning: enabling versioning on a bucket, creating multiple versions of the same object, deleting/restoring, and configuring lifecycle rules.

In [ ]:
import boto3
from datetime import datetime

# Connect to RustFS S3
s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
)

# Create bucket for this lab
s3.create_bucket(Bucket='lab5-versions')
print("Bucket created.")

In [ ]:
# Enable versioning on the bucket
s3.put_bucket_versioning(
    Bucket='lab5-versions',
    VersioningConfiguration={'Status': 'Enabled'}
)
print("Versioning enabled!")

In [ ]:
# Upload version 1 of document.txt
response = s3.put_object(
    Bucket='lab5-versions',
    Key='document.txt',
    Body=b'This is version 1 of the document.'
)
version_id = response['VersionId']

# Read it back to confirm
get_response = s3.get_object(Bucket='lab5-versions', Key='document.txt')
content = get_response['Body'].read().decode('utf-8')
print(f"Version 1 content: {content}")
print(f"Version ID: {version_id}")

In [ ]:
# Upload version 2 (overwrite)
response = s3.put_object(
    Bucket='lab5-versions',
    Key='document.txt',
    Body=b'This is version 2 of the document.'
)
version_id = response['VersionId']

# Read it back to confirm
get_response = s3.get_object(Bucket='lab5-versions', Key='document.txt')
content = get_response['Body'].read().decode('utf-8')
print(f"Version 2 content: {content}")
print(f"Version ID: {version_id}")

In [ ]:
# Upload version 3 (overwrite)
response = s3.put_object(
    Bucket='lab5-versions',
    Key='document.txt',
    Body=b'This is version 3 of the document.'
)
version_id = response['VersionId']
print(f"Version 3 uploaded with Version ID: {version_id}")

In [ ]:
# List all versions of the object
response = s3.list_object_versions(Bucket='lab5-versions')
print("All versions:")
for version in response.get('Versions', []):
    print(f"  VersionId: {version['VersionId']}")
    print(f"  IsLatest: {version['IsLatest']}")
    print(f"  LastModified: {version['LastModified']}")
    print(f"  Content: {version.get('Key', 'N/A')}")
    print()

In [ ]:
# Delete the object (creates a delete marker)
s3.delete_object(Bucket='lab5-versions', Key='document.txt')
print("Object deleted.")

In [ ]:
# Try reading current version — should fail due to delete marker
try:
    s3.get_object(Bucket='lab5-versions', Key='document.txt')
    print("Object still accessible (unexpected)")
except Exception as e:
    print(f"Current read failed as expected: {e}")

# Get the oldest version ID and read it directly
versions = s3.list_object_versions(Bucket='lab5-versions')
oldest_version = versions['Versions'][-1]['VersionId']
response = s3.get_object(
    Bucket='lab5-versions',
    Key='document.txt',
    VersionId=oldest_version
)
restored_content = response['Body'].read().decode('utf-8')
print(f"Restored version 1 content: {restored_content}")

In [ ]:
# Permanently delete all versions
response = s3.list_object_versions(Bucket='lab5-versions')

# Delete all versions
for version in response.get('Versions', []):
    s3.delete_object(
        Bucket='lab5-versions',
        Key=version['Key'],
        VersionId=version['VersionId']
    )
    print(f"Deleted version {version['VersionId']}")

# Delete delete markers
for marker in response.get('DeleteMarkers', []):
    s3.delete_object(
        Bucket='lab5-versions',
        Key=marker['Key'],
        VersionId=marker['VersionId']
    )
    print(f"Deleted marker {marker['VersionId']}")

# Verify bucket is empty
remaining = s3.list_object_versions(Bucket='lab5-versions')
assert len(remaining.get('Versions', [])) == 0
assert len(remaining.get('DeleteMarkers', [])) == 0
print("All versions purged.")

In [ ]:
# Delete the bucket
s3.delete_bucket(Bucket='lab5-versions')
print("Deleted bucket lab5-versions")
print("Cleanup done!")